In [1]:
import os

In [2]:
import pandas as pd

In [3]:
from dotenv import load_dotenv, find_dotenv

In [4]:
from azure.search.documents import SearchClient

In [5]:
from azure.search.documents.models import VectorizableTextQuery

In [6]:
from azure.core.credentials import AzureKeyCredential

In [7]:
load_dotenv(find_dotenv(usecwd=True), override=True)

True

In [8]:
AZURE_SEARCH_SERVICE_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_INDEX = "rag-1791093456903"
AZURE_SEARCH_ADMIN_KEY = os.getenv("AZURE_SEARCH_KEY")

In [9]:
print("Endpoint:", AZURE_SEARCH_SERVICE_ENDPOINT)
print("Índice:", AZURE_SEARCH_INDEX)
print("Clave:", "✅" if AZURE_SEARCH_ADMIN_KEY else "❌ FALTA")

Endpoint: https://search-rag-platzi-25797.search.windows.net
Índice: rag-1791093456903
Clave: ✅


In [10]:
search_client = SearchClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX,
    credential=AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY),
)

In [11]:
def display_results(results):
    df = pd.json_normalize(list(results)).dropna(axis=1, how='all')
    df["chunk"] = df["chunk"].apply(lambda c: c[:300] + '...' if len(c) > 300 else c)
    first_cols = ['title', 'chunk', '@search.score']
    df = df[first_cols + [col for col in df.columns if col not in first_cols]]

    df = df.style.set_properties(**{
        'max-width': '500px',
        'text-align': 'left',
        'white-space': 'normal',
        'word-wrap': 'break-word'
    }).hide(axis="index")


    return df

In [12]:
results = search_client.search(search_text="What is Contoso", top=5, select=["title", "chunk"])

In [13]:
display_results(results)

title,chunk,@search.score
Northwind_Standard_Benefits_Details.pdf,"your coverage, please contact our customer service department at Contoso. Neurodevelopmental Therapy (Habilitation) Neurodevelopmental Therapy (Habilitation) At Contoso, we want to ensure that our employees and their dependents receive the best possible care and services. That’s why we’ve ...",3.737474
Northwind_Health_Plus_Benefits_Details.pdf,"the tips outlined above, you can help ensure that your request for services or treatments is approved in a timely manner and that you are receiving the most appropriate care. The Group And You OTHER INFORMATION ABOUT THIS PLAN The Group and You The Northwind Health Plus plan is a gro...",3.706240
Northwind_Standard_Benefits_Details.pdf,"At Contoso, we understand that medical costs can be intimidating and confusing, which is why we’ve partnered with Northwind Health to offer our employees the Northwind Standard plan. This plan provides a balance billing protection, meaning that you are protected from unexpected costs when visi...",3.591514
Northwind_Health_Plus_Benefits_Details.pdf,"potential side effects that may occur from these treatments, and to make sure that you are comfortable with the potential risks and benefits of the treatment. Chemotherapy And Radiation Therapy Chemotherapy and Radiation Therapy: Covered Services At Contoso, we are proud to offer our emplo...",3.508141
Northwind_Standard_Benefits_Details.pdf,"providers that are not available from participating providers. Additionally, in some cases, the health plan may cover non-participating providers’ charges if there are no participating providers in your area. Tips In order to avoid costly balance billing amounts, it is important to make sure...",3.151717


In [14]:
results = search_client.search(vector_queries=[VectorizableTextQuery(text="What is Contoso", k_nearest_neighbors=50, fields="text_vector")], top=5, select=["title", "chunk"])

In [15]:
display_results(results)

title,chunk,@search.score
MSFT_cloud_architecture_contoso.pdf,How a fictional but representative global organization has implemented the Microsoft Cloud Contoso in the Microsoft Cloud This topic is 1 of 7 in a series The Contoso Corporation Contoso s worldwide organization Elements of Contoso s implementation of the Microsoft cloud Networking Net...,0.702329
role_library.pdf,"be responsible for providing excellent customer service to all Contoso customers, addressing inquiries and resolving customer issues. Responsibilities: • Maintain a professional and courteous attitude when interacting with customers via phone, email, and chat • Respond to custome...",0.702326
MSFT_cloud_architecture_contoso.pdf,have identified the following elements when planning for the adoption of Microsoft s cloud offerings. Microsoft Cloud Identity for Enterprise Architects Microsoft Cloud Identity for Enterprise Architects Microsoft Cloud Networking for Enterprise Architects Microsoft Cloud Networking for...,0.690531
role_library.pdf,"to Contoso Electronics’ financial operations, ensuring the financial health of the organization. The CFO will lead all financial operations, including accounting, financial planning and analysis, budgeting, forecasting, financial reporting, and risk management. Responsibilities: • D...",0.687055
MSFT_cloud_architecture_contoso.pdf,https://technet.microsoft.com/library/mt775341.aspx https://technet.microsoft.com/library/mt775341.aspx How a fictional but representative global organization has implemented the Microsoft Cloud Contoso in the Microsoft Cloud Contoso s IT infrastructure and needs Mapping Contoso s busin...,0.686630


In [16]:
results = search_client.search(
    search_text="What is Contoso",
    vector_queries=[VectorizableTextQuery(text="What is Contoso", k_nearest_neighbors=50, fields="text_vector")],
    top=5,
    select=["title", "chunk"]
)

In [17]:
display_results(results)

title,chunk,@search.score
MSFT_cloud_architecture_contoso.pdf,"scenarios How a fictional but representative global organization has implemented the Microsoft Cloud 1 2 3 4 5This topic is 7 of 7 in a series 6 With the networking, identity, and security infrastructure in place, Contoso began to address its business needs with enterprise cloud scenarios. ...",0.027497
employee_handbook.pdf,"Contoso Electronics, we believe in maintaining a safe and transparent working environment for all of our team members. To ensure the well-being of the entire organization, we have established a Whistleblower Policy. This policy encourages employees to come forth and report any unethical or ill...",0.026974
MSFT_cloud_architecture_contoso.pdf,How a fictional but representative global organization has implemented the Microsoft Cloud Contoso in the Microsoft Cloud This topic is 1 of 7 in a series The Contoso Corporation Contoso s worldwide organization Elements of Contoso s implementation of the Microsoft cloud Networking Net...,0.026190
MSFT_cloud_architecture_contoso.pdf,have identified the following elements when planning for the adoption of Microsoft s cloud offerings. Microsoft Cloud Identity for Enterprise Architects Microsoft Cloud Identity for Enterprise Architects Microsoft Cloud Networking for Enterprise Architects Microsoft Cloud Networking for...,0.025475
MSFT_cloud_architecture_contoso.pdf,https://technet.microsoft.com/library/mt775341.aspx https://technet.microsoft.com/library/mt775341.aspx How a fictional but representative global organization has implemented the Microsoft Cloud Contoso in the Microsoft Cloud Contoso s IT infrastructure and needs Mapping Contoso s busin...,0.025429
